In [1]:
import os

In [2]:
%pwd

'c:\\Users\\hasan\\End to end\\AutoML\\notebooks'

In [3]:
os.chdir("C:/Users/hasan/End to end/AutoML")
%pwd

'C:\\Users\\hasan\\End to end\\AutoML'

In [5]:
import pandas as pd 

df = pd.read_csv(r"test_data\Titanic-Dataset.csv")

In [28]:
report = {
            "status": "valid",
            "dataset_summary": {},
            "target_analysis" : {},
            "column_analysis": {},
            "actions": {},
            "problems": [],
            "recommendations" : []
        }

In [29]:
report["dataset_summary"] =  {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "file_size_mb": round(os.path.getsize(r"test_data\Titanic-Dataset.csv") / (1024 * 1024), 2)
    }

In [30]:
print(report)

{'status': 'valid', 'dataset_summary': {'rows': 891, 'columns': 12, 'file_size_mb': 0.06}, 'target_analysis': {}, 'column_analysis': {}, 'actions': {}, 'problems': [], 'recommendations': []}


In [31]:
target = "Survived"
problem_type = "classification"

if target not in df.columns:
    target_info = {
        "name": target,
        "type": None,
        "class_balance": None,
        "imbalance_flag": True,
        "error": f"Target column '{target}' not found"
    }

else:
    target_series = df[target].dropna()

    if problem_type == "regression":
        target_info = {
            "name": target,
            "type": "numeric",
            "class_balance": None,
            "imbalance_flag": False
        }

    elif problem_type == "classification":
        class_distribution = target_series.value_counts(normalize=True).to_dict()
        
        target_info = {
            "name": target,
            "type": "categorical",
            "class_balance": {str(k): round(v, 4) for k, v in class_distribution.items()},
        }

report["target_analysis"] = target_info

In [32]:
print(report)

{'status': 'valid', 'dataset_summary': {'rows': 891, 'columns': 12, 'file_size_mb': 0.06}, 'target_analysis': {'name': 'Survived', 'type': 'categorical', 'class_balance': {'0': 0.6162, '1': 0.3838}}, 'column_analysis': {}, 'actions': {}, 'problems': [], 'recommendations': []}


In [ ]:
column_analysis = {}

for col in df.columns:
    series = df[col]

    dtype = str(series.dtype)
    missing_pct = series.isna().mean()
    unique_values = series.nunique(dropna=True)

    column_analysis[col] = {
            "dtype": dtype,
            "missing_pct": float(missing_pct),
            "unique_values": int(unique_values),
        }

report["column_analysis"] = column_analysis

In [39]:
print(report)

{'status': 'valid', 'dataset_summary': {'rows': 891, 'columns': 12, 'file_size_mb': 0.06}, 'target_analysis': {'name': 'Survived', 'type': 'categorical', 'class_balance': {'0': 0.6162, '1': 0.3838}}, 'column_analysis': {'PassengerId': {'dtype': 'int64', 'missing_pct': 0.0, 'unique_values': 891}, 'Survived': {'dtype': 'int64', 'missing_pct': 0.0, 'unique_values': 2}, 'Pclass': {'dtype': 'int64', 'missing_pct': 0.0, 'unique_values': 3}, 'Name': {'dtype': 'object', 'missing_pct': 0.0, 'unique_values': 891}, 'Sex': {'dtype': 'object', 'missing_pct': 0.0, 'unique_values': 2}, 'Age': {'dtype': 'float64', 'missing_pct': 19.865319865319865, 'unique_values': 88}, 'SibSp': {'dtype': 'int64', 'missing_pct': 0.0, 'unique_values': 7}, 'Parch': {'dtype': 'int64', 'missing_pct': 0.0, 'unique_values': 7}, 'Ticket': {'dtype': 'object', 'missing_pct': 0.0, 'unique_values': 681}, 'Fare': {'dtype': 'float64', 'missing_pct': 0.0, 'unique_values': 248}, 'Cabin': {'dtype': 'object', 'missing_pct': 77.1043771

In [42]:
one_hot_threshold = 3

drop_columns = [col for col, stats in report["column_analysis"].items() if stats["missing_pct"] > 20]
impute_columns = [col for col, stats in column_analysis.items() 
                      if 0 < stats["missing_pct"] <= 20]
    
remove_duplicates = df.duplicated().sum() > 0
    
one_hot = []
label_encode = []
    
for col, stats in column_analysis.items():
    if stats["dtype"] in ["object", "category"]:
        if stats["unique_values"] <= one_hot_threshold:
            one_hot.append(col)
        else:
                label_encode.append(col)
    
actions = {
        "drop_columns": drop_columns,
        "impute_columns": impute_columns,
        "remove_duplicates": bool(remove_duplicates),
        "encoding":
            {
                "one_hot" : one_hot,
                "label_encode" : label_encode
            }
    }

print(actions)

{'drop_columns': ['Cabin'], 'impute_columns': ['Age', 'Embarked'], 'remove_duplicates': False, 'encoding': {'one_hot': ['Sex', 'Embarked'], 'label_encode': ['Name', 'Ticket', 'Cabin']}}
